# Statistical Properties 09 — Structural Breaks

**Goal:** Detect structural breaks in MTG card price series — sudden, persistent level shifts driven by events such as card bans, unbans, set releases, or tournament results. STL decomposition (Stat 03) captures smooth seasonal patterns but cannot model sudden jumps.

**Tables:** gold_price_features (full history per card)

**Methods:**
- Bai-Perron test (multiple structural breaks) — `ruptures` library
- Chow test for known break dates (ban/unban event calendar)
- CUSUM test — detects parameter instability in a rolling regression

**Why this matters:**
- Ban/unban events create instant price jumps of 50–800%
- Structural breaks violate stationarity assumptions — a card series may appear I(1) when it's actually I(0) with a break
- The model needs a is_price_spike feature (already in gold_price_features) and possibly a post-ban indicator

⚠️ **Data requirement:** Bai-Perron requires ≥30 observations per series. With 3 snapshots: fully deferred.

In [1]:
import duckdb
import numpy as np
import pandas as pd

In [2]:
gold = duckdb.connect("../../data/gold/cards.duckdb", read_only=True)

In [3]:
# Load full history including is_price_spike: that flag is already in gold_price_features
# and serves as a pre-computed signal for break-point candidates, which the ruptures
# algorithm can later validate against the actual time series shape
df = gold.execute("""
    SELECT uuid, snapshot_date, eur, is_price_spike
    FROM gold_price_features
    WHERE eur IS NOT NULL AND uuid IS NOT NULL
    ORDER BY uuid, snapshot_date
""").df()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["log_eur"] = np.log1p(df["eur"])

n_snapshots = df["snapshot_date"].nunique()
# 30 observations minimum: Bai-Perron requires enough data on both sides of each break
# candidate to estimate stable pre-break and post-break means; below 30, the algorithm
# tends to place spurious breaks at the endpoints of the series
MIN_OBS = 30

print(f"Total cards:    {df['uuid'].nunique():,}")
print(f"Snapshots:      {n_snapshots}  (need ≥{MIN_OBS} for Bai-Perron)")
print(f"Price spikes:   {df['is_price_spike'].sum():,} rows flagged")
if n_snapshots < MIN_OBS:
    rerun = (
        df["snapshot_date"].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)
    ).date()
    print("\n⚠ INSUFFICIENT DATA — all tests deferred.")
    print(f"  Re-run after approximately {rerun} (≥{MIN_OBS} daily snapshots)")

Total cards:    82,413
Snapshots:      33  (need ≥30 for Bai-Perron)
Price spikes:   0 rows flagged


## 1. Bai-Perron Multiple Structural Break Detection

**Method:** `ruptures.Binseg` (binary segmentation) on log1p(EUR) per card. Detects up to k=5 break points using BIC to select k.

**Why Bai-Perron:** It detects multiple breaks simultaneously, unlike the Chow test which requires a known break date. Particularly useful for cards with multiple ban/unban events (Oko, Uro, etc. have been banned in multiple formats).

**Output:** For each card with ≥30 observations, number of structural breaks detected. Aggregate: what fraction of cards have at least 1 break?

In [4]:
if n_snapshots < MIN_OBS:
    print(f"DEFERRED: {n_snapshots} snapshots (need ≥{MIN_OBS}).")
    print()
    print("What this section will compute:")
    print("  For each card with ≥30 observations:")
    print("    ruptures.Binseg(model='rbf').fit_predict(log_eur, n_bkps=5)")
    print("    Select n_bkps via BIC minimization")
    print("  Report: fraction of cards with ≥1 break, distribution of break counts")
    print()
    print("Expected results:")
    print("  ~5–15% of cards expected to have at least 1 structural break")
    print(
        "  High-break cards: banned/unbanned cards (Oko, Uro, Companion mechanic etc.)"
    )
    print(
        "  The is_price_spike flag in gold_price_features is a proxy for break detection"
    )
    print()
    print("Retest schedule:")
    print(
        f"  Bai-Perron: ≥{MIN_OBS} snapshots (~{(df['snapshot_date'].max() + pd.Timedelta(days=MIN_OBS - n_snapshots)).date()})"
    )
else:
    try:
        import ruptures as rpt
    except ImportError:
        print("ruptures not installed. Install with: pip install ruptures")
        print("Falling back to CUSUM test only.")
        rpt = None

    if rpt is not None:
        obs_per_card = df.groupby("uuid").size()
        testable = obs_per_card[obs_per_card >= MIN_OBS].index
        break_results = []
        for uuid, group in df[df["uuid"].isin(testable)].groupby("uuid"):
            series = group["log_eur"].dropna().values
            if len(series) < MIN_OBS:
                continue
            if np.std(series) == 0:
                # A perfectly flat price series has no structural break by definition --
                # skip Binseg rather than let a zero-variance penalty force spurious splits.
                break_results.append(
                    {"uuid": uuid, "n_obs": len(series), "n_breaks": 0}
                )
                continue
            # Binseg (binary segmentation): recursively splits the series at the point
            # that maximises the reduction in within-segment variance;
            # model='rbf' uses a radial basis function kernel that detects mean shifts
            algo = rpt.Binseg(model="rbf").fit(series)
            # BIC-style penalty: lets the algorithm choose how many breaks the data
            # actually justifies, instead of forcing the same fixed count (min(5, n//10))
            # on every card regardless of whether a real break is present.
            penalty = np.log(len(series)) * np.var(series)
            bkps = algo.predict(pen=penalty)
            # predict() returns segment END indices; subtract 1 for the break count
            n_breaks = len(bkps) - 1
            break_results.append(
                {"uuid": uuid, "n_obs": len(series), "n_breaks": n_breaks}
            )

        df_breaks = pd.DataFrame(break_results)
        frac_broken = (df_breaks["n_breaks"] >= 1).mean()
        print(f"Cards tested: {len(df_breaks):,}")
        print(
            f"Cards with ≥1 break: {(df_breaks['n_breaks'] >= 1).sum():,} ({frac_broken * 100:.1f}%)"
        )
        print(df_breaks["n_breaks"].value_counts().sort_index().to_string())

Cards tested: 79,908
Cards with ≥1 break: 1,086 (1.4%)
n_breaks
0    78822
6     1086


## 2. Event-Anchored Break Detection (Chow Test)

**Method:** For known ban/unban dates, test whether the mean of log_eur changes before vs after the event (two-sample t-test or Mann-Whitney U).

**Event calendar (examples):**
- Companion mechanic nerf: 2020-06-01 (errata changed rules, not ban)
- Oko, Thief of Crowns ban: 2019-11-18 (banned in Standard, Pioneer, Modern)
- Uro, Titan of Nature's Wrath ban: 2020-09-28

**Why event-anchored:** Bai-Perron detects breaks but can't label them. Event-anchored tests confirm whether a known event caused a break — useful for feature engineering (add a post-ban indicator).

In [5]:
if n_snapshots < MIN_OBS:
    print(f"DEFERRED: {n_snapshots} snapshots (need ≥{MIN_OBS}).")
    print()
    print("This section tests known ban/unban event dates:")
    print(
        "  For each event date, compare mean log_eur in (date-30d, date) vs (date, date+30d)"
    )
    print("  Mann-Whitney U test; p < 0.05 → break confirmed at that event date")
    print()
    # Event calendar will live in gold_events once the ingestion pipeline adds it;
    # for now the test logic is documented here and will activate automatically
    print(
        "Requires an event calendar table (to be loaded from a future gold_events table)"
    )
    print("or hardcoded ban dates for specific high-profile cards (Oko, Uro, etc.)")
else:
    # Once the event calendar is available, iterate over (card_uuid, event_date) pairs
    # and compare the 30-day window before vs after each event using MWU
    print("Chow test implementation available once event calendar is loaded.")
    print("See gold_events table (to be implemented in pipeline).")

Chow test implementation available once event calendar is loaded.
See gold_events table (to be implemented in pipeline).


In [6]:
gold.close()

## 📋 Final Conclusions

```
DATA
─────────────────────────────────────────────────────────────────────────────
Snapshots: 3  (need ≥30 for Bai-Perron)
All tests: DEFERRED

BAI-PERRON STRUCTURAL BREAKS
─────────────────────────────────────────────────────────────────────────────
Status: DEFERRED — re-run after ≥30 snapshots (~2026-07-04)
Expected: 5–15% of cards have ≥1 detectable structural break
High-break cards: banned cards (Oko, Uro), unban events, set-release staples

EVENT-ANCHORED CHOW TEST
─────────────────────────────────────────────────────────────────────────────
Status: DEFERRED — requires event calendar table (gold_events)
Known events: Companion nerf, Oko ban, Uro ban, various Format bans
Expected: ban events → CONFIRMED break; rule changes → ambiguous

MODEL IMPLICATIONS
─────────────────────────────────────────────────────────────────────────────
If structural breaks confirmed:
  1. Add is_price_spike (already in gold_price_features) as a model feature
  2. Add days_since_last_ban / days_since_last_unban (requires event calendar)
  3. Consider separate model training windows (pre-break vs post-break)
  4. Tier 3 model especially sensitive to structural breaks (ban events dominate)

RETEST SCHEDULE
─────────────────────────────────────────────────────────────────────────────
Bai-Perron per card:  ≈ 2026-07-04  (≥30 snapshots)
Chow test events:     requires gold_events table (pipeline feature)
```